<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 05 — The Report

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook assembles the deliverable for Ex_05: a short markdown report in
which you choose an architecture and defend the choice.

## What is being marked

Two questions carry most of the marks, one from each lecture of block L5.

> **L5.1.** The AC power flow is the ground truth. The graph network
> approximates it: on the six-bus network it is about twice as fast for one case
> and about a hundred times faster per case in a batch, and its answer survives a
> renumbering of the buses — but it missed the undervoltage after line 1-3
> tripped. **The graph network answers far faster than the AC power flow, and
> less exactly. When is the speed worth the error, before and after a line
> trips?**

> **L5.2.** Persistence, a recurrent network and an LSTM forecast the same
> substation load. **Which would you deploy on a substation controller, and
> which number or plot decided it?**

Neither has a fixed correct answer. Both have a correct *shape* of answer: a
claim, the number that supports it, and the condition under which it would stop
being true. An answer with a number and no condition scores less than one with
both, even when the number is the same.

Then the **four questions from L3.2 slide 2**, asked about the model you chose:

1. What is the input, precisely?
2. What is the loss — what single number was minimised?
3. Where did the data come from, and who paid for it?
4. What happens when it is wrong?

These four are how every exercise report in this course is marked.

## How to use this notebook

Run notebooks 01, 03 and 04 first — this one loads the `.npz` files they wrote
and refuses to build a report without them. Then fill in the `ANSWERS`
dictionary, run the rest, and check the printed report before submitting it.

---

## 0 · Load what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_5_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np

import Ex_5_core as core

needed = {"nb01": "nb01_cnn.npz",
          "nb03": "nb03_gnn.npz",
          "nb04": "nb04_sequence.npz"}

results = {}
missing = []
core.needed('nb01_cnn.npz', 'nb03_gnn.npz', 'nb04_sequence.npz')   # on Colab without Drive, asks for the missing files
for key, filename in needed.items():
    path = os.path.join(core.OUTPUT_DIR, filename)
    if os.path.exists(path):
        results[key] = np.load(path, allow_pickle=True)
        print(f"loaded {filename}")
    else:
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        "missing " + ", ".join(missing)
        + " — run notebooks 01, 03 and 04 before this one")

---

## 0b · Your personal seed

Every notebook in this exercise set fixes the seed to 0, so the printed
"what you should see" blocks are true on any machine. That is right for
checking your work and wrong for reporting it — with one seed the whole cohort
produces identical numbers.

So the numbers below are **yours**. Put your study number in, run the cell, and
quote what it prints in your answers where the questions ask for it. Your
supervisor can regenerate exactly these numbers from your study number alone,
so they are worth getting right and pointless to invent.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = core.personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own batch of radiographs, from the same procedural process.
X_you, y_you = core.weld_images(n_per_class=20, seed=SEED)
MEANS_YOU = [float(X_you[y_you == k].mean()) for k in range(len(core.CLASS_NAMES))]

print()
print(f"  your batch : {X_you.shape}")
for name, m in zip(core.CLASS_NAMES, MEANS_YOU):
    print(f"    mean pixel value, {name:<6s} : {m:.5f}")

**What you should see.** Above the seed cell, three `loaded ...` lines. If you
get a `FileNotFoundError`, go back and run the notebook it names, all the way to
the cell that saves. The seed cell prints your study number, your seed, and the
mean pixel value of each class in your own batch of radiographs.

---

## 1 · The evidence, gathered

Everything you measured, in one place. Read it before you write anything: the
report is meant to be an argument from these numbers, not an essay decorated with
them.

In [ ]:
nb1, nb3, nb4 = results["nb01"], results["nb03"], results["nb04"]

print("=" * 66)
print("NOTEBOOK 01 — a CNN against a dense network, on weld radiographs")
print("=" * 66)
print(core.error_table(
    [["CNN", f"{int(nb1['n_cnn']):,}", f"{float(nb1['acc_train']):.3f}",
      f"{float(nb1['acc_test']):.3f}"],
     ["dense", f"{int(nb1['n_mlp']):,}", f"{float(nb1['acc_train_mlp']):.3f}",
      f"{float(nb1['acc_test_mlp']):.3f}"]],
    ["model", "parameters", "train accuracy", "held-out accuracy"]))
print(f"\none layer, 16 x 16 image -> eight 16 x 16 maps: dense {int(nb1['dense_16']):,}"
      f"  vs convolution {int(nb1['conv_16']):,} parameters")

print()
print("=" * 66)
print("NOTEBOOK 03 — the power flow on six buses")
print("=" * 66)
print(core.error_table(
    [["graph network", f"{int(nb3['n_gnn']):,}",
      f"{float(nb3['rmse_theta_gnn']):.5f}", f"{float(nb3['rmse_volt_gnn']):.5f}"],
     ["dense network", f"{int(nb3['n_mlp']):,}",
      f"{float(nb3['rmse_theta_mlp']):.5f}", f"{float(nb3['rmse_volt_mlp']):.5f}"]],
    ["model (error against the AC power flow)", "parameters", "angle RMSE [rad]",
     "|V| RMSE [p.u.]"]))
t_ac = float(nb3["t_ac"])
print("\nspeed on the six-bus network, per case:")
print(core.error_table(
    [["AC power flow (Newton-Raphson)", f"{1e3 * t_ac:.4f}", "1"],
     ["graph network, one case", f"{1e3 * float(nb3['t_gnn_single']):.4f}",
      f"{t_ac / float(nb3['t_gnn_single']):.1f}"],
     ["graph network, all held-out cases at once", f"{1e3 * float(nb3['t_gnn_batched']):.4f}",
      f"{t_ac / float(nb3['t_gnn_batched']):.0f}"]],
    ["method", "time per case [ms]", "times faster than the AC power flow"]))
print("\nspeed on larger synthetic grids (graph network with random weights):")
print(core.error_table(
    [[int(n), f"{1e3 * ta:.3f}", f"{1e3 * ts:.3f}", f"{1e3 * tb:.4f}",
      f"{ta / ts:.1f}", f"{ta / tb:.0f}"]
     for n, ta, ts, tb, _ in nb3["speed_sweep"]],
    ["buses", "AC power flow [ms]", "GNN, one case [ms]", "GNN, batch [ms per case]",
     "faster, one case", "faster, batch"]))
print("\nbuses renumbered:")
print(core.error_table(
    [["graph network", f"{float(nb3['gap_gnn']):.1e}", f"{float(nb3['rmse_gnn_perm']):.5f}"],
     ["dense network", f"{float(nb3['gap_mlp']):.1e}", f"{float(nb3['rmse_mlp_perm']):.5f}"]],
    ["model", "largest change in the answer [rad]", "angle RMSE [rad], renumbered"]))
print("\nline 1-3 tripped, no retraining (error against a new AC power flow):")
print(core.error_table(
    [["graph network (given the new A)", f"{float(nb3['rmse_theta_gnn_trip']):.5f}",
      f"{float(nb3['rmse_volt_gnn_trip']):.5f}"],
     ["dense network (cannot be told)", f"{float(nb3['rmse_theta_mlp_trip']):.5f}",
      f"{float(nb3['rmse_volt_mlp_trip']):.5f}"]],
    ["model", "angle RMSE [rad]", "|V| RMSE [p.u.]"]))
print("\ndepth sweep:")
print(core.error_table([[int(d), f"{int(n):,}", f"{rt:.5f}", f"{rv:.5f}"]
                        for d, n, rt, rv in nb3["depth_results"]],
                       ["layers", "parameters", "angle RMSE [rad]", "|V| RMSE [p.u.]"]))

print()
print("=" * 66)
print("NOTEBOOK 04 — forecasting the next hour of load")
print("=" * 66)
mse_p = float(nb4["mse_persistence"])
print(core.error_table(
    [["persistence", "0", f"{mse_p:.6f}", "1.00"]]
    + [[str(n), f"{int(p):,}", f"{v:.6f}", f"{v / mse_p:.2f}"]
       for n, p, v in zip(nb4["names"], nb4["params"], nb4["val"])],
    ["model", "parameters", "held-out MSE [p.u.^2]", "vs persistence"]))

**What you should see.** The tables you saw in notebooks 01, 03 and 04, in one
place. Nothing new — but seeing them together is what makes the report writable,
because the argument runs across them.

Three observations to take into the writing.

**Where the assumption is true, the constrained model wins.** A CNN assumes a
defect means the same wherever it is; a graph network assumes a bus's physics
does not depend on its number. Both assumptions hold here, and both models were
the more accurate on held-out data with fewer parameters: 0.989 against 0.711 on
the radiographs, 0.00057 against 0.00078 rad on the angles. Only the graph
network's answer survived a renumbering of the buses.

**Fast is not the same as right.** The AC power flow is the ground truth; the
graph network is an approximation of it that answers far faster — about twice as
fast for one six-bus case, about a hundred times per case in a batch, and on the
larger synthetic grids faster one case at a time from about a hundred buses and
in a batch at every size. But after the line trip both networks' angle errors
grew about a hundredfold, to 0.052 and 0.078 rad, and both missed voltages below
0.95 p.u. that the power flow found. When the speed is worth that error is the
L5.1 question.

**A table is not the whole forecast.** Both networks beat persistence, to 0.39
and 0.27 of its error, and the LSTM beat the recurrent network with 3.8 times the
parameters — at one seed, and not yet converged. The forecast plot shows where
the difference comes from. That is the L5.2 question.

---

## 2 · Your answers

Fill in the dictionary. Write in full sentences; the strings become the report.

Length guidance: two to four sentences per entry for the four questions, and a
paragraph each for the two marked questions. A worked example of the expected
standard is printed further down.

In [ ]:
ANSWERS = {
    # ---- the model you are defending ------------------------------------
    "model": "",           # e.g. "the three-layer graph network from notebook 03"

    # ---- L5.1: accuracy against speed -------------------------------------
    "l51": "",
    # The graph network answers far faster than the AC power flow, and less
    # exactly. When is the speed worth the error, before and after a line
    # trips? Use the accuracy, the timing and the tripped-line numbers, and say
    # what would have to be true before you used the network in place of the
    # power flow.

    # ---- L5.2: which forecast would you deploy? ------------------------------
    "l52": "",
    # Persistence, the recurrent network or the LSTM: which would you deploy
    # on a substation controller, and which number or plot decided it? Say what
    # you would check before trusting the ranking — one seed, 300 epochs.

    # ---- the four questions from L3.2 slide 2 ---------------------------
    "q1_input": "",        # what is the input, precisely? shapes and units
    "q2_loss": "",         # what single number was minimised, and in what units
    "q3_data": "",         # where did the data come from, and who paid for it
    "q4_wrong": "",        # what happens when it is wrong

    # ---- one thing you would do next ------------------------------------
    "next": "",            # one experiment, and what it would settle
}

blank = [k for k, v in ANSWERS.items() if not v.strip()]
print("filled in:", len(ANSWERS) - len(blank), "of", len(ANSWERS))
if blank:
    print("still empty:", ", ".join(blank))

**What you should see.** `filled in: 8 of 8` once you are done. Until then it
lists what is missing.

---

## 3 · A worked example, so the standard is visible rather than guessed

This is an answer to question 4 — *what happens when it is wrong* — for the
six-bus graph network. It is not the only good answer, and it is not about your
model. It is here to show the level of specificity that scores well.

> **What happens when it is wrong.** The model estimates bus angles to about
> 0.0006 radians (0.03 degrees) and voltage magnitudes to about 0.0003 per unit
> on held-out cases drawn from the same operating envelope. A wrong estimate has two distinct
> consequences depending on where it is wrong. At a metered bus the error is
> visible: the operator can compare against the phasor measurement and discard
> the model. At an unmetered bus it is not, and an angle error of a few
> hundredths of a radian on a heavily loaded corridor is enough to misjudge the
> direction of a marginal flow and dispatch generation the wrong way. The failure
> is silent — nothing about the model's output announces it — which is why the
> estimate at an unmetered bus must be labelled as an inference and not as a
> reading, in every plot handed to somebody who did not run the code. The
> operating envelope is the second failure mode: the training injections came
> from a fixed range on one topology, and when line 1-3 tripped the angle error
> rose ninety-fold and the model missed voltages below 0.95 p.u. at three buses.
> Anything outside that envelope is extrapolation, and the model gives no
> indication that it has left it.

Note what makes that answer work: it names the error in physical units, separates
detectable from undetectable failure, names the consequence to somebody outside
the room, and identifies the condition under which the numbers stop applying. An
answer of the form "the predictions would be inaccurate" scores near zero.

---

## 4 · Build the report

In [ ]:
lines = []
lines.append("# Ex_05 report — CNN and GNN\n")
lines.append("*Deep Learning for Engineering, Part 1. Model defended: "
             + (ANSWERS["model"] or "**not stated**") + ".*\n")

lines.append("\n## Results\n")
lines.append("### A CNN against a dense network (notebook 01)\n")
lines.append(core.error_table(
    [["CNN", f"{int(nb1['n_cnn']):,}", f"{float(nb1['acc_train']):.3f}",
      f"{float(nb1['acc_test']):.3f}"],
     ["dense", f"{int(nb1['n_mlp']):,}", f"{float(nb1['acc_train_mlp']):.3f}",
      f"{float(nb1['acc_test_mlp']):.3f}"]],
    ["model", "parameters", "train accuracy", "held-out accuracy"]))

lines.append("\n### The power flow on six buses (notebook 03)\n")
lines.append(core.error_table(
    [["graph network", f"{int(nb3['n_gnn']):,}",
      f"{float(nb3['rmse_theta_gnn']):.5f}", f"{float(nb3['rmse_gnn_perm']):.5f}",
      f"{float(nb3['rmse_theta_gnn_trip']):.5f}"],
     ["dense network", f"{int(nb3['n_mlp']):,}",
      f"{float(nb3['rmse_theta_mlp']):.5f}", f"{float(nb3['rmse_mlp_perm']):.5f}",
      f"{float(nb3['rmse_theta_mlp_trip']):.5f}"]],
    ["model", "parameters", "angle RMSE [rad] against the AC power flow",
     "buses renumbered", "line 1-3 tripped"]))
lines.append("\nVoltage magnitude, RMSE [p.u.]: graph network "
             f"{float(nb3['rmse_volt_gnn']):.5f} intact, {float(nb3['rmse_volt_gnn_trip']):.5f} tripped; "
             f"dense network {float(nb3['rmse_volt_mlp']):.5f} intact, "
             f"{float(nb3['rmse_volt_mlp_trip']):.5f} tripped.\n")
lines.append("\nTime per case, graph network against the AC power flow (CPU):\n")
lines.append(core.error_table(
    [["6 (the trained network)", f"{1e3 * float(nb3['t_ac']):.4f}",
      f"{1e3 * float(nb3['t_gnn_single']):.4f}", f"{1e3 * float(nb3['t_gnn_batched']):.4f}"]]
    + [[f"{int(n)} (synthetic)", f"{1e3 * ta:.4f}", f"{1e3 * ts:.4f}", f"{1e3 * tb:.4f}"]
       for n, ta, ts, tb, _ in nb3["speed_sweep"] if int(n) in (48, 96, 768)],
    ["buses", "AC power flow [ms]", "GNN, one case [ms]", "GNN, batch [ms per case]"]))

lines.append("\n### Forecasting the next hour of load (notebook 04)\n")
lines.append(core.error_table(
    [["persistence", "0", f"{float(nb4['mse_persistence']):.6f}", "1.00"]]
    + [[str(n), f"{int(p):,}", f"{v:.6f}", f"{v / float(nb4['mse_persistence']):.2f}"]
       for n, p, v in zip(nb4["names"], nb4["params"], nb4["val"])],
    ["model", "parameters", "held-out MSE [p.u.^2]", "vs persistence"]))

lines.append("\n## L5.1 — when is the speed worth the error?\n")
lines.append(ANSWERS["l51"] or "*not answered*")
lines.append("\n\n## L5.2 — which forecast would you deploy?\n")
lines.append(ANSWERS["l52"] or "*not answered*")

lines.append("\n\n## The four questions (L3.2 slide 2)\n")
for question, key in zip(core.four_questions(),
                         ["q1_input", "q2_loss", "q3_data", "q4_wrong"]):
    lines.append(f"\n**{question}**\n\n" + (ANSWERS[key] or "*not answered*") + "\n")

lines.append("\n## What I would do next\n")
lines.append(ANSWERS["next"] or "*not answered*")
lines.append("\n")

report = "\n".join(lines)

os.makedirs(core.OUTPUT_DIR, exist_ok=True)
report_path = os.path.join(core.OUTPUT_DIR, "Ex05_report.md")
with open(report_path, "w", encoding="utf-8") as fh:
    fh.write(report)

print("wrote", report_path, f"({len(report)} characters)")
print("\n" + "=" * 70 + "\n")
print(report)

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 04

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 17, each under its question, to the end of the
report you just wrote. The last one is not from a notebook: it is the
question across all of them, and it concludes the report. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 04 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · Finding Weld Defects with a CNN ----------------------
    # 01.1 The first convolution of your CNN has 80 parameters; a dense layer
    # making the same eight 16 by 16 feature maps would need 526,336. Where
    # does each number come from, and why does the image size appear in one
    # count but not in the other? What property of the weld images makes the
    # smaller count safe to use? (-> L5.1 Q3, Q4)
    "01.1": """
""",
    # 01.2 Follow one radiograph through your CNN and write down its shape
    # after every layer. What does the padding do to the size, what do the two
    # pooling layers do, and what does pooling throw away that a classifier of
    # clean, crack and pit does not need? (-> L5.1 Q2, Q3)
    "01.2": """
""",
    # 01.3 Your hand-set feature maps were drawn in red and blue, although the
    # radiograph is grey. What is a feature map, and what do red, blue and
    # white mean on it? Why did the blob kernel, chosen by hand to find pits,
    # hardly separate pits from clean plate, and what does training do instead?
    # (-> L5.1 Q1)
    "01.3": """
""",
    # 01.4 Both networks fitted their training images almost perfectly, yet on
    # the held-out images the CNN scored 0.989 with 2,019 parameters and the
    # dense network 0.711 with 16,643. Explain the difference to a colleague
    # who says more parameters give a better model. What did the CNN not save,
    # and when would the dense network be the better choice? (-> L5.1 Q4)
    "01.4": """
""",

    # ---- notebook 02 · Graphs from Scratch ----------------------------------
    # 02.1 The adjacency matrix of the six-bus network records that a line
    # exists but not its impedance. Name a prediction on this network where
    # leaving the impedances out would matter, and say how the line data could
    # be given to a graph network alongside the adjacency. (-> L5.1 Q6)
    "02.1": """
""",
    # 02.2 In section 3 a value placed at bus 0 spread one hop per round and
    # then flattened towards a common value. Say what one round of message
    # passing does, what over-smoothing is, and how the graph's diameter of
    # three sets the smallest number of layers a graph network on this network
    # needs. (-> L5.1 Q7, Q8)
    "02.2": """
""",
    # 02.3 The permutation check gave a gap of exactly zero with random,
    # untrained weights. Say in your own words what permutation equivariance
    # guarantees, and why its holding for random weights shows that it comes
    # from the layer rather than from training. (-> L5.1 Q9)
    "02.3": """
""",
    # 02.4 The message-passing layer has 24 weights however many buses there
    # are, while a dense layer on the same buses has 456 at six buses and over
    # four million at six hundred. Explain where the difference comes from, and
    # relate it to weight sharing in a convolution on an image. (-> L5.1 Q9)
    "02.4": """
""",

    # ---- notebook 03 · Node Regression on a Six-Bus Network -----------------
    # 03.1 A power-flow bus is slack, PV or PQ. For each type, say which two of
    # P, Q, |V| and theta are given and which two are solved, and explain why
    # the slack bus's |V| and theta did not change across the 800 cases. Why
    # must the bus type reach the graph network as a feature rather than as a
    # bus number? (-> L5.1 Q6)
    "03.1": """
""",
    # 03.2 The graph network and the dense network reached similar held-out
    # errors, but renumbering the buses made the dense network's angle error
    # about sixty times larger and left the graph network's unchanged. Explain
    # why, and say which of the two you would hand to a grid operator. (-> L5.1
    # Q9)
    "03.2": """
""",
    # 03.3 In the depth sweep the held-out error was largest with one layer and
    # grew again with six. Explain both ends of the sweep using the graph's
    # diameter and over-smoothing. (-> L5.1 Q8)
    "03.3": """
""",
    # 03.4 The graph network only approximates the AC power flow it was trained
    # on, yet it answered many cases far faster, most of all in a batch. With
    # line 1-3 tripped it missed voltages below 0.95 p.u. that the power flow
    # found. When is the speed worth having, and what would have to be true
    # before you used the network to screen contingencies? (-> L5.1 Q10)
    "03.4": """
""",

    # ---- notebook 04 · Sequences: a Recurrent Network and an LSTM -----------
    # 04.1 The recurrent network and the LSTM both read the same 24-hour
    # window, with a hidden size of 16 and the same training recipe. Say what
    # the hidden state is and how the recurrent network updates it hour by
    # hour. Why does its parameter count of 321 not depend on the window
    # length, and why does what it saw early in the window fade by the last
    # hour? (-> L5.2 Q2, Q3, Q4)
    "04.1": """
""",
    # 04.2 The LSTM has 1,233 parameters and the recurrent network 321. Account
    # for the difference from the LSTM's equations: name the three gates and
    # the candidate, and say what each one decides. Explain how the addition in
    # the cell-state update lets the LSTM keep a longer memory than the
    # recurrent network. Did the extra parameters pay for themselves on this
    # problem? (-> L5.2 Q7, Q9)
    "04.2": """
""",
    # 04.3 Persistence has no parameters, and both networks were judged against
    # it. Why is it the right baseline for a one-hour-ahead forecast, and what
    # would a network with a held-out MSE above 0.00332 tell you? In the
    # forecast plot, where does persistence go wrong, and why do the networks
    # do better exactly there? (-> L5.2 Q5, Q10)
    "04.3": """
""",
    # 04.4 The windows were split in time order and never shuffled. Explain
    # what goes wrong with a shuffled split when two neighbouring windows share
    # twenty-three of their twenty-four hours. Then say what else you would
    # check before claiming that the LSTM is better than the recurrent network,
    # given that changing the seed moved both networks' errors. (-> L5.2 Q3,
    # Q10)
    "04.4": """
""",

    # ---- to conclude, across all the notebooks ------------------------------
    # C Across a grid, a graph and a sequence: when did building the data's
    # structure into the network beat a dense network, and what did it cost?
    # (-> L5.1 and L5.2, Exercise slide)
    "C": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'Finding Weld Defects with a CNN', 'The first convolution of your CNN has 80 parameters; a dense layer making the same eight 16 by 16 feature maps would need 526,336. Where does each number come from, and why does the image size appear in one count but not in the other? What property of the weld images makes the smaller count safe to use?', 'L5.1 Q3, Q4'),
    "01.2": ('01', 'Finding Weld Defects with a CNN', 'Follow one radiograph through your CNN and write down its shape after every layer. What does the padding do to the size, what do the two pooling layers do, and what does pooling throw away that a classifier of clean, crack and pit does not need?', 'L5.1 Q2, Q3'),
    "01.3": ('01', 'Finding Weld Defects with a CNN', 'Your hand-set feature maps were drawn in red and blue, although the radiograph is grey. What is a feature map, and what do red, blue and white mean on it? Why did the blob kernel, chosen by hand to find pits, hardly separate pits from clean plate, and what does training do instead?', 'L5.1 Q1'),
    "01.4": ('01', 'Finding Weld Defects with a CNN', 'Both networks fitted their training images almost perfectly, yet on the held-out images the CNN scored 0.989 with 2,019 parameters and the dense network 0.711 with 16,643. Explain the difference to a colleague who says more parameters give a better model. What did the CNN not save, and when would the dense network be the better choice?', 'L5.1 Q4'),
    "02.1": ('02', 'Graphs from Scratch', 'The adjacency matrix of the six-bus network records that a line exists but not its impedance. Name a prediction on this network where leaving the impedances out would matter, and say how the line data could be given to a graph network alongside the adjacency.', 'L5.1 Q6'),
    "02.2": ('02', 'Graphs from Scratch', "In section 3 a value placed at bus 0 spread one hop per round and then flattened towards a common value. Say what one round of message passing does, what over-smoothing is, and how the graph's diameter of three sets the smallest number of layers a graph network on this network needs.", 'L5.1 Q7, Q8'),
    "02.3": ('02', 'Graphs from Scratch', 'The permutation check gave a gap of exactly zero with random, untrained weights. Say in your own words what permutation equivariance guarantees, and why its holding for random weights shows that it comes from the layer rather than from training.', 'L5.1 Q9'),
    "02.4": ('02', 'Graphs from Scratch', 'The message-passing layer has 24 weights however many buses there are, while a dense layer on the same buses has 456 at six buses and over four million at six hundred. Explain where the difference comes from, and relate it to weight sharing in a convolution on an image.', 'L5.1 Q9'),
    "03.1": ('03', 'Node Regression on a Six-Bus Network', "A power-flow bus is slack, PV or PQ. For each type, say which two of P, Q, |V| and theta are given and which two are solved, and explain why the slack bus's |V| and theta did not change across the 800 cases. Why must the bus type reach the graph network as a feature rather than as a bus number?", 'L5.1 Q6'),
    "03.2": ('03', 'Node Regression on a Six-Bus Network', "The graph network and the dense network reached similar held-out errors, but renumbering the buses made the dense network's angle error about sixty times larger and left the graph network's unchanged. Explain why, and say which of the two you would hand to a grid operator.", 'L5.1 Q9'),
    "03.3": ('03', 'Node Regression on a Six-Bus Network', "In the depth sweep the held-out error was largest with one layer and grew again with six. Explain both ends of the sweep using the graph's diameter and over-smoothing.", 'L5.1 Q8'),
    "03.4": ('03', 'Node Regression on a Six-Bus Network', 'The graph network only approximates the AC power flow it was trained on, yet it answered many cases far faster, most of all in a batch. With line 1-3 tripped it missed voltages below 0.95 p.u. that the power flow found. When is the speed worth having, and what would have to be true before you used the network to screen contingencies?', 'L5.1 Q10'),
    "04.1": ('04', 'Sequences: a Recurrent Network and an LSTM', 'The recurrent network and the LSTM both read the same 24-hour window, with a hidden size of 16 and the same training recipe. Say what the hidden state is and how the recurrent network updates it hour by hour. Why does its parameter count of 321 not depend on the window length, and why does what it saw early in the window fade by the last hour?', 'L5.2 Q2, Q3, Q4'),
    "04.2": ('04', 'Sequences: a Recurrent Network and an LSTM', "The LSTM has 1,233 parameters and the recurrent network 321. Account for the difference from the LSTM's equations: name the three gates and the candidate, and say what each one decides. Explain how the addition in the cell-state update lets the LSTM keep a longer memory than the recurrent network. Did the extra parameters pay for themselves on this problem?", 'L5.2 Q7, Q9'),
    "04.3": ('04', 'Sequences: a Recurrent Network and an LSTM', 'Persistence has no parameters, and both networks were judged against it. Why is it the right baseline for a one-hour-ahead forecast, and what would a network with a held-out MSE above 0.00332 tell you? In the forecast plot, where does persistence go wrong, and why do the networks do better exactly there?', 'L5.2 Q5, Q10'),
    "04.4": ('04', 'Sequences: a Recurrent Network and an LSTM', "The windows were split in time order and never shuffled. Explain what goes wrong with a shuffled split when two neighbouring windows share twenty-three of their twenty-four hours. Then say what else you would check before claiming that the LSTM is better than the recurrent network, given that changing the seed moved both networks' errors.", 'L5.2 Q3, Q10'),
    "C": ('C', 'to conclude', "Across a grid, a graph and a sequence: when did building the data's structure into the network beat a dense network, and what did it cost?", 'L5.1 and L5.2, Exercise slide'),
}

report_md = os.path.join(core.OUTPUT_DIR, "Ex05_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex05_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += ["### To conclude" if nb == "C" else f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex05_report.md: 17 of 17 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex05_report.md into Ex05_report.pdf, with any figure
# saved as Ex05_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(core.OUTPUT_DIR, "Ex05_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open(os.path.join(core.OUTPUT_DIR, "Ex05_report.md"), encoding="utf-8").read()
figs = sorted(glob.glob("Ex05_report*.png")
              + glob.glob(os.path.join(core.OUTPUT_DIR, "Ex05_report*.png")))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download(pdf_path)
except ImportError:
    pass


**What you should see.** `wrote .../Ex05_outputs/Ex05_report.md`, then the
report printed in full. Read it. If a section says *not answered*, it will say
that to the marker too.

---

## 5 · Optional extensions

None of these is required. Each is about half an hour and each settles a question
the notebooks left open.

**Hold the parameter count fixed in the depth sweep.** Notebook 03's sweep
changed depth and parameter count together, so part of the change from one layer
to three is ordinary capacity rather than reach. Shrink the hidden width as the
depth grows — the equal-budget comparison of L4.2 — and see whether three layers
stay the best.

**Train the graph network across topologies.** Notebook 03 showed that being able
to *accept* a new adjacency matrix is not enough to survive a line trip. Build a
training set that mixes the intact network with several single-line trips,
feeding the correct $A$ with each case, and test on a trip held out from
training. This is the "one model, many grids" formulation, and it is the
experiment that would justify the architecture properly.

**Give the graph network the line data.** Its layer uses $A$ only, so a short
line and a long one count the same. Weight each neighbour by its line's
admittance instead of by 1, and see what it buys — on the intact network and
after the trip.

**Time a trained network on a larger grid.** Notebook 03's speed sweep used
networks with random weights, because a forward pass costs the same whatever the
weights. Train one on a synthetic grid of a hundred buses or more and see whether
it keeps the six-bus accuracy at the speed the sweep promised.

**Break the translation assumption in notebook 01.** Restrict the defects to the
upper-left quadrant of the plate and retrain both models. The convolution's
advantage should shrink, and the size of the shrinkage is a measurement of how
much the architectural assumption was worth.

**Test notebook 04's ranking.** Neither network had converged at 300 epochs, and
changing the seed moved both errors. Train both for 600 epochs from three seeds
and see whether the LSTM's lead over the recurrent network survives.

---

## 6 · What Ex_05 was for

Three kinds of data, one idea, and one habit.

**The idea** is L5.1's opening sentence: a learned local rule, applied
everywhere. A convolution applies it on a grid, message passing applies it on a
graph, a recurrent network applies it along a line of time. In every case the
parameter count is set by the *rule* and not by the size of the thing it is
applied to, and in every case the sharing encodes an assumption about the
problem — that position does not matter, that bus numbering does not matter,
that the same rule serves every hour.

**The habit** is to ask what the architecture is claiming and then check the
claim against the data. Notebook 01's claim was true, and the CNN won outright.
Notebook 03's claim was true too: the graph network came close to the AC power
flow, answered far faster, and was the only learned model that survived a
renumbering — but it had learned one topology, and when a line tripped it missed
the undervoltage the power flow found. Notebook 04's claim held, and both networks beat
persistence; whether the LSTM's gates were worth their cost on a 24-hour window
is a question one seed cannot settle.
